In [26]:
!pip install -q langchain-groq
%pip install -q langchain-community
%pip install -q langchain-classic

In [28]:
import os, requests
from google.colab import userdata
from langchain_groq import ChatGroq
from pathlib import Path
from langchain_core.prompts import (ChatPromptTemplate, 
                                    HumanMessagePromptTemplate,
                                    )
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_classic.chains.llm import LLMChain

In [16]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [17]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    model_kwargs= {"seed" : 350 },
    temperature= 0,
    max_tokens= 250
)

chatmessage history

In [48]:
background_info = ChatMessageHistory()
background_info.add_user_message('whats the biggest planet of our solar system ?')
background_info.add_ai_message('Jupiter. It’s more than twice as massive as all the other planets in the solar system combined')

In [52]:
human_template = HumanMessagePromptTemplate.from_template('{question}')

chat_template = ChatPromptTemplate.from_messages( background_info.messages + [human_template])

print(chat_template.messages)

[HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}), AIMessage(content='Jupiter. It’s more than twice as massive as all the other planets in the solar system combined', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [53]:
chain = LLMChain(
    llm = llm,
    prompt= chat_template
)

In [54]:
response =  chain.invoke({'question' : 'how wide it is than earth ?'})

print(response['text'])

**Jupiter vs. Earth – a quick size comparison**

| Property | Earth | Jupiter | How many times bigger? |
|----------|-------|---------|------------------------|
| **Mean diameter** | **12 742 km** (≈7 930 mi) | **139 822 km** (≈86 881 mi) | **≈11 ×** Earth’s diameter |
| **Equatorial diameter** | 12 756 km | 142 984 km | **≈11.2 ×** |
| **Polar diameter** | 12 714 km | 133 708 km | **≈10.5 ×** |
| **Mean radius** | 6 371 km | 69 911 km | **≈
